In [1]:
%matplotlib qt
from neuronol_power import *

In [2]:
# Read preprocessed EEG data
fmax = 170 # Hz
data_path = '~/data/sample'
raw_file = op.join(data_path, 'sample_audvis_eeg.fif')
raw = mne.io.read_raw_fif(raw_file)
raw.crop(tmin=216, tmax=276) # pseudo-resting data interval

params = {}
params['T_max'] = 60
params['sfreq'] = np.floor(raw.info['sfreq'])
params['size_epoch'] = 1.2
params['epoch_times'] = np.arange(-119, 601) / params['sfreq']
params['bipolar_connections'] = load_bipolar_connections('bipolar_connections.pickle')

Opening raw data file ~/data/sample/sample_audvis_eeg.fif...
    Read a total of 1 projection items:
        Average EEG reference (1 x 60)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.


In [10]:
# Frequency bands
freq_bands = {
    'Delta': [0.5, 4],
    'Theta': [4, 8],
    'Alpha': [8, 12],
    'Beta': [12, 30],
    'Gamma': [30, fmax],
    'Range': [0.5, fmax],
    'M1-Delta': [1.56, 3.12],
    'M1-Theta': [3.9, 7.02],
    'M1-Alpha': [7.8, 12.48],
    'M1-LowBeta': [13.26, 18.72],
    'M1-HighBeta': [19.5, 39.78],
    'M1-Range': [1.56, 39.78],
    'M2-Beta1+Beta2': [12.5, 20.0],
    'M2-Beta3': [20.5, 28.0],
    'M3-Delta': [0.5, 5.5],
    'M3-Theta': [6.0, 8.0],
    'M3-Alpha': [8.5, 12.0],
    'M3-Beta1': [12.5, 18.0],
    'M3-Beta3': [21.0, 30.0],
    'M3-Range': [0.5, 30.0],
    'M4-Band1': [1.3, 3.5],
    'M4-Band2': [3.5, 7.5],
    'M4-Band3': [7.5, 10.5],
    'M4-Band4': [10.5, 13],
    'M4-Band5': [13, 16],
    'M4-Band6': [16, 20],
    'M4-Band7': [20, 25],
    'M4-Band8': [25, 30],
    'M4-Band9': [30, 35],
    'M4-Delta+Theta': [1.3, 7.5],
    'M4-Alpha': [7.5, 13],
    'M4-Beta': [13, 35],
    'M4-Range': [1.3, 35],
}

In [4]:
channel_sets = {
    'MNE-Frontal': np.arange(1, 17),
    'MNEBipolar-Frontal': np.array([0, 1, 2, 12, 13, 14, 19, 20, 24, 25, 30, 31,
                                    32, 42, 43, 44, 49, 50, 54, 55, 60, 61]),
}

# Method-1: Bauer 2001
Non-overlapping EEG epochs were detrended and cosine tapered. Then, the EEG epochs were submitted to a Fast Fourier Transform which computed the power (in $\mu V^2$) of the EEG. The power spectra were averaged over epochs.

Estimates of power within the frequency bands were expressed as a proportion of total power across the entire frequency range, i.e. Relative Power (RP). These relative power estimates for each frequency band were averaged across all the electrode sites.

<ins>Note</ins>: No reference is specified.

__Frequency band(s)__ = [1.56 - 3.12]  [3.90 - 7.02]  [7.80 - 12.48]  [13.26 - 18.72]  [19.50 - 39.78]

__Measurement(s)__ = $RP_{mean, all channels}^{HighBeta}$

__Total measurement(s)__ = 1 band $\times$ 1 measurement = 1

__Total feature(s)__ = __Total measurement(s)__ = 1

In [5]:
psds_mean_epochs_M1, freqs = psd_M1(params, raw, plot_psd_epochs=None)
F1_M1 = featureFunc_bandRelativePower(psds_mean_epochs_M1, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)
print('F1_M1 = %0.3f.' % F1_M1)

A total of 60 seconds of raw EEG data has been studied.
Fs = 600.0
Each epoch has 1.2 seconds of EEG data.
Shape of times = (720,)
Shape of raw_data = (60, 36000)
Total number of epochs studied = 50
Shape of epochs = (50, 60, 720)
Shape of psds = (50, 60, 359)
F1_M1 = 0.093.


qt.qpa.drawing: Layer-backing can not be explicitly controlled on 10.14 when built against the 10.14 SDK


# Method-2: Rangaswamy et al. 2002
The data were first transformed into bipolar derivations. Spectral densities were calculated using Fourier transform on overlapping epochs (50% overlap) with Hamming windowing.

The resulting spectral densities were aggregated into bands, divided by the bandwidth, and subsequently averaged across epochs to get the absolute power data. A logarithmic transformation of the values was applied to this data to normalize their distribution.

__Frequency band(s)__

All loci: [12.5 - 20.0]

Frontal: [20.5 - 28.0]

__Measurement(s)__ = $LOG (\frac{AP_{band}}{f_{band, max} - f_{band, min}})$

__Total measurement(s)__ = 2 channel-band groups $\times$ 1 measurement = 2

__Total feature(s)__ = __Total measurement(s)__ = 2

In [6]:
psds_mean_epochs_M2, freqs = psd_M2(params, raw, plot_psd_epochs=None)
F3_M2 = featureFunc_bandAbsolutePower(psds_mean_epochs_M2, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNEBipolar-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)
print('F3_M2 = %0.3f.' % F3_M2)

A total of 60 seconds of raw EEG data has been studied.
Fs = 600.0
Each epoch has 1.2 seconds of EEG data.
Shape of times = (720,)
Reading 0 ... 36037  =      0.000 ...    60.000 secs...
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=65, n_times=36038
    Range : 155533 ... 191570 =    258.956 ...   318.956 secs
Ready.
Added the following bipolar channels:
01-04, 04-08, 08-18, 18-17, 17-25, 25-36, 36-44, 44-53, 53-57, 18-26, 26-37, 37-44, 01-05, 05-09, 09-19, 19-27, 27-45, 45-54, 54-57, 05-10, 10-28, 28-38, 38-46, 46-54, 01-11, 11-20, 20-29, 29-39, 39-47, 47-57, 03-07, 07-16, 16-23, 23-24, 24-35, 35-43, 43-52, 52-56, 56-59, 23-34, 34-42, 42-52, 03-06, 06-15, 15-22, 22-33, 33-51, 51-55, 55-59, 06-14, 14-32, 32-41, 41-50, 50-55, 03-13, 13-21, 21-31, 31-40, 40-49, 49-59, 02-12, 12-30, 30-48, 48-58, 58-60
Shape of raw_data = (65, 36000)
Total number of epochs studied = 99
Shape of epochs = (99, 65, 720)
Shape of psds = (99, 65, 359)
F3_M2 = 4.1

# Method-3: Winterer et al. 1998
Non-overlapping consecutive EEG epochs referenced to auricular activity were extracted and submitted to spectral analysis using the Fast Fourier Transform. 

The square root of absolute power (magnitude), expressed in $\mu V$, as well as the relative power, was computed for all the frequency bands.

<ins>Note</ins>: The sample dataset used does not mention the reference used. __Ensure auricular referencing__ when working with the actual dataset for the study.

__Frequency band(s)__

[0.5 - 5.5]  [6.0 - 8.0]  [8.5 - 12.0]  [12.5 - 18.0]  [21.0 - 30.0]

__Measurement(s)__ = $\sqrt{AP}$; RP; TP

__Total measurement(s)__ = n_channels $\times$ (5 bands $\times$ 2 measurements + 1 measurement)

__Total feature(s)__ = __Total measurement(s)__

In [7]:
psds_mean_epochs_M3, freqs = psd_M3(params, raw, plot_psd_epochs=None)
F14_M3_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)
print('F14_M3_00 = %0.3f.' % F14_M3_00)

A total of 60 seconds of raw EEG data has been studied.
Fs = 600.0
Each epoch has 1.2 seconds of EEG data.
Shape of times = (720,)
Shape of raw_data = (60, 36000)
Total number of epochs studied = 50
Shape of epochs = (50, 60, 720)
Shape of psds = (50, 60, 359)
F14_M3_00 = 73528.870.


In [8]:
Fj_M3 = []
for ich in np.arange(60):
    Fj_M3_ii = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M3-Range'],
                                             channels=[ich], norm_and_log_density=None,
                                             compute_magnitude=None, plot_psd=None)
    Fj_M3.append(Fj_M3_ii)
# Fj_M3

# Method-4: Saletu-Zyhlarz et al. 2004
EEG referenced to the average of mastoids was recorded. Spectral analysis was performed on non-overlapping epochs using the Fast Fourier Transform. The spectral curves were averaged over epochs and channels.

<ins>Note</ins>: 

1) The sample dataset used does not mention the reference used. __Ensure mastoid referencing__ when working with the actual dataset for the study.

2) This method is exactly the same as Method-3 (Winterer et al. 1998) except for mastoid/auricular referencing.

__Frequency band(s)__

[1.3 - 3.5]  [3.5 - 7.5]  [7.5 - 10.5]  [10.5 - 13]  [13 - 16]  [16 - 20]  [20 - 25]  [25 - 30]  [30 - 35]

$\delta + \theta$: [1.3 - 7.5]

$\alpha$: [7.5 - 13]

$\beta$: [13 - 35]

__Measurement(s)__

TP; -- [1]

AP/RP in 12 frequency bands; -- [$2 \times 12 = 24$]

Dominant Frequency (DF, in Hz); -- [1]

AP/RP of DF; -- [$2 \times 1 = 2$]

centre-of-gravity frequencies (centroids) of {$\delta + \theta$, $\alpha$, $\beta$, full-range}; -- [$1 \times 4 = 4$]

standard deviation of centroids of {$\delta + \theta$, $\alpha$, $\beta$, full-range}  -- [$1 \times 4 = 4$]

__Total measurement(s)__ = 1 + 24 + 1 + 2 + 4 + 4 = 36

__Total feature(s)__ = __Total measurement(s)__ = 36

In [11]:
psds_mean_epochs_M4, freqs = psd_M4(params, raw, plot_psd_epochs=None)
F15_M4 = featureFunc_bandAbsolutePower(psds_mean_epochs_M4, freqs, freq_bands['M4-Range'],
                                       channels=None, norm_and_log_density=None,
                                       compute_magnitude=None, plot_psd=True)
print('F15_M4 = %0.3f.' % F15_M4)

A total of 60 seconds of raw EEG data has been studied.
Fs = 600.0
Each epoch has 1.2 seconds of EEG data.
Shape of times = (720,)
Shape of raw_data = (60, 36000)
Total number of epochs studied = 50
Shape of epochs = (50, 60, 720)
Shape of psds = (50, 60, 359)
F15_M4 = 25287.676.


# Method-5: Coutin-Churchman et al. 2003, 2006
EEG activity was recorded in the resting state and the data were referenced to the average of the two auricular reference electrodes. Power spectra from non-overlapping EEG epochs were computed using the Persyst EEG analysis software.

<ins>Note</ins>:

1) Since they used a proprietary software with closed-source methods, we do not know the exact method used for the estimation of power spectra. Therefore, we will use a well-functioning AP calculation method to calulate power and use it to extract all the features/biomarkers highlighted in their study. __Ensure non-overlapping epochs__ in the chosen AP method.

2) The sample dataset used does not mention the reference used. __Ensure auricular referencing__ when working with the actual dataset for the study.

AP and RP of AUD patients were compared with a normative database (composed of data from control subjects) using Z-scores. The measures were logarithmically transformed to achieve gaussianity before estimating the mean and standard deviation used for Z-transforms.

$$
Z_e = \frac{ln(P_e) - \bar{x_e}}{sd_e}
$$

$P_e$ is the given power measure at the e recording electrode site, $\bar{x_e}$ and $sd_e$ the corresponding mean and standard deviation of the same measure (after log-transform) at electrode e, obtained from the control data. An abnormal increase for a given absolute and/or relative power measure was defined as a Z score greater than 3, and an abnormal decrease as a Z score less than 3, in at least two neighboring sites for a given band.

AP power was given precedence over RP power to assign abnormalities. RP alterations without AP alterations were only considered at the affected band(s) only if both AP and RP in other bands were within normal limits.

__Frequency band(s)__

(2006) -> [0.5 - 3.0]  [3.5 - 7.5]  [8.0 - 13.0]  [14.0 - 25.0]

(2003) -> [0.5 - 3.5]  [4.0 - 7.5]  [8.0 - 13.0]  [13.5 - 25.0]

__Measurement(s)__ = AP; RP

__Total measurement(s)__ = n_channels $\times$ 7 bands $\times$ 2 measurements

__Total feature(s)__ = 1 (per subject) $\times$ 7 bands $\times$ 1 (abnormal AP | RP at 2 or more electrode sites) = 7

# Comparison of powers from different methods

In [ ]:
psds_mean_epochs_M1, freqs = psd_M1(params, raw, plot_psd_epochs=None)
psds_mean_epochs_M2, freqs = psd_M2(params, raw, plot_psd_epochs=None)
psds_mean_epochs_M3, freqs = psd_M3(params, raw, plot_psd_epochs=None)
psds_mean_epochs_M4, freqs = psd_M4(params, raw, plot_psd_epochs=None)
psds_mean_epochs_M5, freqs = psd_M5(params, raw, plot_psd_epochs=None)

In [ ]:
# Comparison of feature F1 derived from powers using all methods
F1_M1 = featureFunc_bandRelativePower(psds_mean_epochs_M1, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)
F1_M2 = featureFunc_bandRelativePower(psds_mean_epochs_M2, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)
F1_M3 = featureFunc_bandRelativePower(psds_mean_epochs_M3, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)
F1_M4 = featureFunc_bandRelativePower(psds_mean_epochs_M4, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)
F1_M5 = featureFunc_bandRelativePower(psds_mean_epochs_M5, freqs, freq_bands['M1-HighBeta'],
                                      freq_bands['M1-Range'], channels=None,
                                      norm_and_log_density=None, plot_psd=True)

print('F1_M1 = %0.3f.' % F1_M1)
print('F1_M2 = %0.3f.' % F1_M2)
print('F1_M3 = %0.3f.' % F1_M3)
print('F1_M4 = %0.3f.' % F1_M4)
print('F1_M5 = %0.3f.' % F1_M5)

In [ ]:
# Comparison of feature F2 derived from powers using all methods
F2_M1 = featureFunc_bandAbsolutePower(psds_mean_epochs_M1, freqs, freq_bands['M2-Beta1+Beta2'],
                                      channels=None, norm_and_log_density=True, plot_psd=True)
F2_M2 = featureFunc_bandAbsolutePower(psds_mean_epochs_M2, freqs, freq_bands['M2-Beta1+Beta2'],
                                      channels=None, norm_and_log_density=True, plot_psd=True)
F2_M3 = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M2-Beta1+Beta2'],
                                      channels=None, norm_and_log_density=True, plot_psd=True)
F2_M4 = featureFunc_bandAbsolutePower(psds_mean_epochs_M4, freqs, freq_bands['M2-Beta1+Beta2'],
                                      channels=None, norm_and_log_density=True, plot_psd=True)
F2_M5 = featureFunc_bandAbsolutePower(psds_mean_epochs_M5, freqs, freq_bands['M2-Beta1+Beta2'],
                                      channels=None, norm_and_log_density=True, plot_psd=True)

print('F2_M1 = %0.3f.' % F2_M1)
print('F2_M2 = %0.3f.' % F2_M2)
print('F2_M3 = %0.3f.' % F2_M3)
print('F2_M4 = %0.3f.' % F2_M4)
print('F2_M5 = %0.3f.' % F2_M5)

In [ ]:
# Comparison of feature F3 derived from powers using all methods
F3_M1 = featureFunc_bandAbsolutePower(psds_mean_epochs_M1, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNE-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)
F3_M2 = featureFunc_bandAbsolutePower(psds_mean_epochs_M2, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNEBipolar-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)
F3_M3 = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNE-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)
F3_M4 = featureFunc_bandAbsolutePower(psds_mean_epochs_M4, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNE-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)
F3_M5 = featureFunc_bandAbsolutePower(psds_mean_epochs_M5, freqs, freq_bands['M2-Beta3'],
                                      channels=channel_sets['MNE-Frontal'],
                                      norm_and_log_density=True, plot_psd=True)

print('F3_M1 = %0.3f.' % F3_M1)
print('F3_M2 = %0.3f.' % F3_M2)
print('F3_M3 = %0.3f.' % F3_M3)
print('F3_M4 = %0.3f.' % F3_M4)
print('F3_M5 = %0.3f.' % F3_M5)

In [ ]:
# Comparison of feature F4 derived from powers using all methods
F4_M1_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M1, freqs, freq_bands['M3-Delta'],
                                         channels=[0], norm_and_log_density=False,
                                         compute_magnitude=True, plot_psd=True)
F4_M2_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M2, freqs, freq_bands['M3-Delta'],
                                         channels=[0], norm_and_log_density=False,
                                         compute_magnitude=True, plot_psd=True)
F4_M3_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M3-Delta'],
                                         channels=[0], norm_and_log_density=False,
                                         compute_magnitude=True, plot_psd=True)
F4_M4_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M4, freqs, freq_bands['M3-Delta'],
                                         channels=[0], norm_and_log_density=False,
                                         compute_magnitude=True, plot_psd=True)
F4_M5_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M5, freqs, freq_bands['M3-Delta'],
                                         channels=[0], norm_and_log_density=False,
                                         compute_magnitude=True, plot_psd=True)

print('F4_M1_00 = %0.3f μV.' % F4_M1_00)
print('F4_M2_00 = %0.3f μV.' % F4_M2_00)
print('F4_M3_00 = %0.3f μV.' % F4_M3_00)
print('F4_M4_00 = %0.3f μV.' % F4_M4_00)
print('F4_M5_00 = %0.3f μV.' % F4_M5_00)

In [ ]:
# Comparison of feature F9 derived from powers using all methods
F9_M1_00 = featureFunc_bandRelativePower(psds_mean_epochs_M1, freqs, freq_bands['M3-Delta'],
                                         freq_bands['M3-Range'], channels=[0],
                                         norm_and_log_density=None, plot_psd=False)
F9_M2_00 = featureFunc_bandRelativePower(psds_mean_epochs_M2, freqs, freq_bands['M3-Delta'],
                                         freq_bands['M3-Range'], channels=[0],
                                         norm_and_log_density=None, plot_psd=False)
F9_M3_00 = featureFunc_bandRelativePower(psds_mean_epochs_M3, freqs, freq_bands['M3-Delta'],
                                         freq_bands['M3-Range'], channels=[0],
                                         norm_and_log_density=None, plot_psd=False)
F9_M4_00 = featureFunc_bandRelativePower(psds_mean_epochs_M4, freqs, freq_bands['M3-Delta'],
                                         freq_bands['M3-Range'], channels=[0],
                                         norm_and_log_density=None, plot_psd=False)
F9_M5_00 = featureFunc_bandRelativePower(psds_mean_epochs_M5, freqs, freq_bands['M3-Delta'],
                                         freq_bands['M3-Range'], channels=[0],
                                         norm_and_log_density=None, plot_psd=False)

print('F9_M1_00 = %0.3f.' % F9_M1_00)
print('F9_M2_00 = %0.3f.' % F9_M2_00)
print('F9_M3_00 = %0.3f.' % F9_M3_00)
print('F9_M4_00 = %0.3f.' % F9_M4_00)
print('F9_M5_00 = %0.3f.' % F9_M5_00)

In [ ]:
# Comparison of feature F14 derived from powers using all methods
F14_M1_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M1, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)
F14_M2_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M2, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)
F14_M3_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M3, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)
F14_M4_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M4, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)
F14_M5_00 = featureFunc_bandAbsolutePower(psds_mean_epochs_M5, freqs, freq_bands['M3-Range'],
                                          channels=[0], norm_and_log_density=None,
                                          compute_magnitude=None, plot_psd=None)

print('F14_M1_00 = %0.3f μV2.' % F14_M1_00)
print('F14_M2_00 = %0.3f μV2.' % F14_M2_00)
print('F14_M3_00 = %0.3f μV2.' % F14_M3_00)
print('F14_M4_00 = %0.3f μV2.' % F14_M4_00)
print('F14_M5_00 = %0.3f μV2.' % F14_M5_00)